In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/flan-t5/pytorch/base/4/config.json
/kaggle/input/flan-t5/pytorch/base/4/spiece.model
/kaggle/input/flan-t5/pytorch/base/4/README.md
/kaggle/input/flan-t5/pytorch/base/4/tokenizer.json
/kaggle/input/flan-t5/pytorch/base/4/tf_model.h5
/kaggle/input/flan-t5/pytorch/base/4/tokenizer_config.json
/kaggle/input/flan-t5/pytorch/base/4/pytorch_model.bin
/kaggle/input/flan-t5/pytorch/base/4/model.safetensors
/kaggle/input/flan-t5/pytorch/base/4/special_tokens_map.json
/kaggle/input/flan-t5/pytorch/base/4/.gitattributes
/kaggle/input/flan-t5/pytorch/base/4/flax_model.msgpack
/kaggle/input/flan-t5/pytorch/base/4/generation_config.json


Quelles sont les commandes nécessaires pour mettre à jour l'environnement, assurer la compatibilité des versions de PyTorch et installer les bibliothèques Hugging Face requises pour le Fine-Tuning (PEFT, LoRA) ?

In [2]:
# 1. On met à jour pip
%pip install --upgrade pip

# 2. CRUCIAL : On met à jour torch ET torchvision ENSEMBLE pour qu'ils soient compatibles
%pip install --upgrade torch torchvision torchaudio

# 3. On installe les bibliothèques du TP
%pip install --upgrade datasets transformers peft evaluate rouge_score loralib

# 4. On fixe le problème de Protobuf (au cas où)
%pip install protobuf==3.20.3

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import time
import torch
import evaluate
import pandas as pd
import numpy as np
import gc # Garbage Collector

from datasets import load_dataset
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, GenerationConfig, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType, PeftModel, PeftConfig

# Nettoyage préventif de la mémoire
gc.collect()
torch.cuda.empty_cache()
print("Bibliothèques importées et mémoire nettoyée.")

2025-11-30 21:34:02.239870: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764538442.390047     136 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764538442.432749     136 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Bibliothèques importées et mémoire nettoyée.


Quel code permet de charger le modèle FLAN-T5 en précision réduite (bfloat16) pour économiser la mémoire GPU, ainsi que son tokenizer ?

In [11]:
# 1. Chargement du dataset DialogSum
huggingface_dataset_name = "knkarthick/dialogsum"
dataset = load_dataset(huggingface_dataset_name)
print("Dataset téléchargé.")

# 2. Chargement du modèle
# CORRECTION : On utilise os.path.abspath pour être sûr qu'il comprenne que c'est un dossier
import os
model_path_kaggle = os.path.abspath('/kaggle/input/flan-t5/pytorch/base/4') 

# Si ça échoue encore, on essaie de charger le fichier config.json d'abord pour le forcer
print(f"Chargement du modèle depuis le dossier local : {model_path_kaggle}")

# On vérifie si le dossier existe (juste pour être sûr)
if not os.path.exists(model_path_kaggle):
    print("ERREUR : Le dossier n'existe pas ! Vérifiez le chemin à droite.")
else:
    print("Dossier trouvé.")

# Chargement
original_model = AutoModelForSeq2SeqLM.from_pretrained(model_path_kaggle, local_files_only=True, torch_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(model_path_kaggle, local_files_only=True)

original_model = original_model.to('cuda')
print("Modèle chargé.")

Dataset téléchargé.
Chargement du modèle depuis le dossier local : /kaggle/input/flan-t5/pytorch/base/4
Dossier trouvé.
Modèle chargé.


Quelle fonction est utilisée pour formater les données d'entrée en ajoutant des instructions ("Summarize...") pour guider le modèle ?

In [12]:
def tokenize_function(example):
    start_prompt = 'Summarize the following conversation.\n\n'
    end_prompt = '\n\nSummary: '
    prompt = [start_prompt + dialogue + end_prompt for dialogue in example["dialogue"]]
    
    # On renvoie des listes Python simples (pas de 'pt')
    example['input_ids'] = tokenizer(prompt, padding="max_length", truncation=True, max_length=512).input_ids
    example['labels'] = tokenizer(example["summary"], padding="max_length", truncation=True, max_length=128).input_ids
    
    return example

# Application de la tokenisation
tokenized_datasets = dataset.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns(['id', 'topic', 'dialogue', 'summary'])

# OPTIMISATION : On garde un échantillon (5%) pour que le TP tourne vite
# Cela permet de vérifier que tout marche sans attendre 1h
tokenized_datasets = tokenized_datasets.filter(lambda example, index: index % 20 == 0, with_indices=True)

print(f"Nombre d'exemples d'entraînement : {len(tokenized_datasets['train'])}")

Map:   0%|          | 0/12460 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Filter:   0%|          | 0/12460 [00:00<?, ? examples/s]

Filter:   0%|          | 0/500 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1500 [00:00<?, ? examples/s]

Nombre d'exemples d'entraînement : 623


Comment configurer LoRA avec un rang de 32, ciblant les modules d'attention "q" et "v", et comment l'appliquer au modèle original ?

In [13]:
lora_config = LoraConfig(
    r=32, # Rang de la matrice (puissance de l'adaptation)
    lora_alpha=32,
    target_modules=["q", "v"], # On cible les modules d'attention
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)

# Création du modèle PEFT
peft_model = get_peft_model(original_model, lora_config)
peft_model.print_trainable_parameters()

trainable params: 3,538,944 || all params: 251,116,800 || trainable%: 1.4093


Quels arguments d'entraînement sont utilisés pour gérer la mémoire limitée (petit batch size avec accumulation de gradients) ?

In [14]:
output_dir = f'./peft-dialogue-summary-training-{str(int(time.time()))}'

peft_training_args = TrainingArguments(
    output_dir=output_dir,
    auto_find_batch_size=True,      # Aide à éviter le crash mémoire
    per_device_train_batch_size=4,  # Petit lot pour soulager le GPU
    gradient_accumulation_steps=4,  # On compense le petit lot
    learning_rate=1e-3,             # Taux d'apprentissage plus élevé pour LoRA
    num_train_epochs=3,             # Nombre de tours d'entraînement
    logging_steps=1,
    save_strategy="epoch",          
    save_total_limit=1,             # On ne garde que le dernier modèle (Sauve l'espace disque !)
    report_to="none"                # Désactive WandB pour éviter les demandes de connexion
)
    
peft_trainer = Trainer(
    model=peft_model,
    args=peft_training_args,
    train_dataset=tokenized_datasets["train"],
)

print("Démarrage de l'entraînement...")
peft_trainer.train()

Démarrage de l'entraînement...


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step,Training Loss
1,36.687500
2,35.437500
3,32.343800
4,28.406200
5,25.437500
6,22.062500
7,20.468800
8,19.875000
9,16.937500
10,14.437500


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


TrainOutput(global_step=60, training_loss=7.3455078125, metrics={'train_runtime': 196.5314, 'train_samples_per_second': 9.51, 'train_steps_per_second': 0.305, 'total_flos': 1300130579349504.0, 'train_loss': 7.3455078125, 'epoch': 3.0})

Une fois le modèle entraîné, quel code permet de générer un résumé à partir d'un dialogue de test ?

In [15]:
# Sauvegarde du modèle entraîné
peft_model_path = "./peft-dialogue-summary-checkpoint-local"
peft_trainer.model.save_pretrained(peft_model_path)
tokenizer.save_pretrained(peft_model_path)
print(f"Modèle sauvegardé dans {peft_model_path}")

# Petit test de génération
index = 200
dialogue = dataset['test'][index]['dialogue']
prompt = f"Summarize the following conversation.\n\n{dialogue}\n\nSummary: "

# On prépare l'entrée pour le GPU
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to('cuda')

# On génère
print("-" * 50)
print("Génération du résumé...")
with torch.no_grad():
    outputs = peft_model.generate(input_ids=input_ids, max_new_tokens=200)
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))
print("-" * 50)

Modèle sauvegardé dans ./peft-dialogue-summary-checkpoint-local
--------------------------------------------------
Génération du résumé...
#Person1# is considering upgrading his computer. #Person2# is considering adding a computer program to his computer.
--------------------------------------------------


Comment calculer le score ROUGE sur un échantillon de test pour valider la performance du modèle ?

In [16]:
# --- ÉTAPE FINALE : Évaluation Quantitative (Score ROUGE) ---
import evaluate

# On charge la métrique
rouge = evaluate.load('rouge')

# On prend un petit échantillon de test (10 dialogues)
dialogues_test = dataset['test'][0:10]['dialogue']
summaries_human = dataset['test'][0:10]['summary']

summaries_peft = []

print("Calcul du score de performance (sur 10 exemples)...")

# On passe en mode évaluation (plus rapide, pas d'entraînement)
peft_model.eval()

with torch.no_grad():
    for i, dialogue in enumerate(dialogues_test):
        if i % 2 == 0: print(f"Traitement exemple {i+1}/10...")
        
        prompt = f"Summarize the following conversation.\n\n{dialogue}\n\nSummary: "
        input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to('cuda')
        
        outputs = peft_model.generate(input_ids=input_ids, max_new_tokens=200)
        summary_gen = tokenizer.decode(outputs[0], skip_special_tokens=True)
        summaries_peft.append(summary_gen)

# Calcul du score
results = rouge.compute(predictions=summaries_peft, references=summaries_human, use_aggregator=True, use_stemmer=True)

print("\n--- RÉSULTATS FINAUX (Score ROUGE) ---")
print(results)

Calcul du score de performance (sur 10 exemples)...
Traitement exemple 1/10...
Traitement exemple 3/10...
Traitement exemple 5/10...
Traitement exemple 7/10...
Traitement exemple 9/10...

--- RÉSULTATS FINAUX (Score ROUGE) ---
{'rouge1': 0.3265615283972282, 'rouge2': 0.09186467941263338, 'rougeL': 0.2866264504033266, 'rougeLsum': 0.2896403695470632}
